I'm gonna use the routine imarith of IRAF for for 'cleaning' the echellograms doing:
$$\frac{<image>-<bias>}{(<flat>-<bias>)_N}$$
where $<flat>$ and $<bias>$ are middle values, they are stored in their own folders.\
The $(<flat>-<bias>)$ has been normalized by dividing it for its own hightest value.

Let's first create the middle value of the image:

In [4]:
import glob
import os
from pyraf import iraf

# 0. If you Windows operative system, be sure that in your input file_list.txt there is _ and not :
# also replace the middle point "·" with "_" on the files automatically dowloaded by the ESO archive into C or D Windows hard disks.

# 1. Load the necessary IRAF packages
iraf.noao()
iraf.imred()
iraf.ccdred()

# 2. Make sure to adjust the pattern to match your actual image file names
image_pattern = "XSHOO*.fits"
image_files = sorted(glob.glob(image_pattern))

if not image_files:
    raise FileNotFoundError(f"No files found with pattern: {image_pattern}")

# If the data comes from MEF (multi-extension) instruments like X-Shooter,
# specify the scientific extension for each file (e.g., bias.fits[1])
# bias_files = [f"{f}[1]" for f in bias_files]

input_list = "image_list.txt"
with open(input_list, "w") as f:   #it's writing file names into the list automatically 
    for filename in image_files:
        f.write(f"{filename}\n")

# 3. Name of the output Master image file
output_master_image = "image.fits"

# Remove the previous output if it already exists
if os.path.exists(output_master_image):
    os.remove(output_master_image)

# 4. Run zerocombine using PyRAF
# Note: combine='median' or 'average' with reject='sigclip' or 'minmax'
iraf.zerocombine(
    input=f"@{input_list}",
    output=output_master_image,
    combine="average",
    reject="minmax",
    ccdtype="",   # Leave it empty to avoid filtering on ccdtype header, the default is "zero" but this fits header has different keywords
    process="no",  # Do not process the output (e.g., do not trim or scale)
    scale="none",  # Bias frames typically do not require scaling
    statsec="",  # Use the entire frame if empty
    #interactive="no",
)

print(f"Master image created successfully: {output_master_image}")

# 5. Cleanup (optional)
#if os.path.exists(input_list):
#    os.remove(input_list)

imred/:
 argus/         ctioslit/       hydra/          kpnocoude/      vtel/
 bias/          dtoi/           iids/           kpnoslit/
 ccdred/        echelle/        irred/          quadred/
 crutil/        generic/        irs/            specred/
ccdred/:
 badpiximage    ccdlist         combine         mkillumcor      setinstrument
 ccdgroups      ccdmask         darkcombine     mkillumflat     zerocombine
 ccdhedit       ccdproc         flatcombine     mkskycor
 ccdinstrument  ccdtest/        mkfringecor     mkskyflat
Master image created successfully: image.fits


Let's now subtract the bias from the image:

In [5]:
import os
from pyraf import iraf

# 1. Load packages containing imarith and statistics
iraf.images()
iraf.imutil()

# 2.0 Define relative paths
bias_dir = "bias"
flat_dir = "flat"

# 2.1 Define file names
image_raw = "image.fits"
master_bias = os.path.join(bias_dir, "Bias.fits")
raw_flat = os.path.join(flat_dir, "Flat.fits")

# Intermediate and final files
flat_sub_bias = os.path.join(flat_dir, "flat_sub_bias.fits")
flat_norm = os.path.join(flat_dir, "flat_norm.fits")
image_sub_bias = "image_sub_bias.fits"
image_calibrated = "image_reduced.fits"


def remove_if_exists(filepath):
    if os.path.exists(filepath):
        os.remove(filepath)


for f in [flat_sub_bias, flat_norm, image_sub_bias, image_calibrated]:
    remove_if_exists(f)

#you have to use APSUM before other things.

# 3. Step 1: Subtraction of Bias from the Flat (Flat - Bias)
iraf.imarith(
    operand1=raw_flat,
    op="-",
    operand2=master_bias,
    result=flat_sub_bias,
    title="Flat debiased",
)

## 4. Step 2: Normalize the debiased Flat by dividing by its MAXIMUM value
#max_val = iraf.imstat(flat_sub_bias, fields="max", format="no", Stdout=1)[
#    0
#].strip()
#print(f"Maximum value found in debiased flat: {max_val}")
#
#iraf.imarith(
#    operand1=flat_sub_bias,
#    op="/",
#    operand2=float(max_val),
#    result=flat_norm,
#    title="Debiased Flat normalized by Max",
#)

# 5. Step 3: Subtraction of Bias from Science (Sci - Bias)
iraf.imarith(
    operand1=image_raw,
    op="-",
    operand2=master_bias,
    result=image_sub_bias,
    title="Image without Bias",
)

## 6. Step 4: Division of debiased Science by normalized Flat ((Sci - Bias) / Flat_norm)
#iraf.imarith(
#    operand1=image_sub_bias,
#    op="/",
#    operand2=flat_norm,
#    result=image_calibrated,
#    title="Image calibrated (Debiased Sci / Max-Norm Flat)",
#)
#
## 7. Cleanup intermediate files (optional)
## for f in [flat_sub_bias, flat_norm, image_sub_bias]:
##     remove_if_exists(f)
#
#print(f"Calibration completed: {image_calibrated}")

I wanna now separate echellogram peaks using APSUM Iraf package:

In [6]:
##it's better to use terminal for apsum, you need to write and press "enter" key. 
#import os
#from pyraf import iraf
#
## 1. Load required spectroscopy packages
#iraf.noao()
#iraf.twodspec()
#iraf.apextract()
#
## 2. Define input and output file names
#input_image = "image_sub_bias.fits"
#output_spectrum = "spec_extracted.ms.fits"
#reference_image = ""  # Leave empty to detect apertures directly on input_image
#
#if os.path.exists(output_spectrum):
#    os.remove(output_spectrum)
#
## 3. Configure and execute apsum in non-interactive batch mode
#iraf.apsum(
#    input=input_image,
#    output="",#output_spectrum,
#    apertur="",  # Use default aperture definitions if reference is empty
#    format="multispec",  # Standard 1D multi-order/extension format
#    references="",#reference_image,  # Template image to inherit trace definitions
#    # Interactivity controls (disabled to prevent terminal hangs)
#    interactive="yes",
#    find="yes",  # Automatically find apertures if reference is empty
#    recenter="no",  # Recenter apertures on current profile peaks
#    resize="no",  # Do not auto-resize aperture widths
#    edit="yes",  # Do not open the interactive graphical editor
#    trace="yes",  # Trace apertures across the dispersion direction
#    fittrace="yes",  # Auto-fit polynomial trace without interactive prompt
#    extract="yes",  # Perform the 1D sum extraction
#    review="yes",  # Do review extractions interactively
#    # Geometry and dispersion sampling
#    line="INDEF",  # Central line/column used to detect spatial profile
#    nsum=10,  # Number of dispersion lines to sum for profile detection
#    # Sky background subtraction
#    background="none",  # 'none', 'average', or 'fit'
#    # Extraction weighting
#    weights="none",  # 'none' for simple sum, 'variance' for optimal extraction
#    clean="no",  # Cosmic ray cleaning (requires valid gain and readnoise)
#)
#
##print(f"Extraction completed successfully: {output_spectrum}")

To separate the orders of your fit image it's better to use iraf terminal and interactive windows.\
Run the apsum task writing an input name "file.fits" and output name like "file.ms.fits", than find the apertures by eye and number them in the correct way (select an high number of apertures at the begin, than press "." and "D" to delate them in the interactive window, "o" to rename them and "M" to add a new one).\
Now you have to fit the peaks found for every order with a polynomial, choose the order writing ":order n" (where n is a number high enough to decrease the RMS but small enough to don't include in your fit random photons that don't rapresent the image of the star). The fit can be iterated multiple times to reject the points we don't need, set the number of iteration writing ":niter n".\
You can save and exit pressing "q".\
\
For this spectra I mostly used polynomial of $4^{th}$ or $5^{th}$ order and a number of iteration between 2 and 5. The RMS was about 0.1, except the last order.\
For flat I used instead $5^th$ and $6th$ order but with higher RMS.\
I found 14 apertures.

I normalize the debiased ms Flat by dividing by its MAXIMUM value. 
For this I used on Iraf terminal "apnormalize" task (with input flat_sub_bias.ms.fits and output flat_norm.ms.fits).

We can now divide the image sub-biased by the flat normalized and sub-biased (both in the .ms.fits  format):

In [ ]:
# Intermediate and final files
flat_sub_bias_ms = os.path.join(flat_dir, "flat_sub_bias.ms.fits")
flat_norm_ms = os.path.join(flat_dir, "flat_norm.ms.fits")
image_sub_bias_ms = "image_sub_bias.ms.fits"
image_calibrated_ms = "image_reduced.ms.fits"


#for f in [flat_sub_bias_ms, flat_norm_ms, image_sub_bias_ms, image_calibrated_ms]:
#    remove_if_exists(f)   #don't remove, files aren't created on this notebook
remove_if_exists(image_calibrated_ms)  #this one is created on this notebook.


# Division of debiased Science by normalized Flat ((Sci - Bias) / Flat_norm)
iraf.imarith(
    operand1=image_sub_bias_ms,
    op="/",
    operand2=flat_norm_ms,
    result=image_calibrated_ms,
    title="Image calibrated (Debiased Sci / Max-Norm Flat)",
)

# Cleanup intermediate files (optional)
# for f in [flat_sub_bias_ms, flat_norm_ms, image_sub_bias_ms]:
#     remove_if_exists(f)

print(f"Calibration multispectrum completed: {image_calibrated_ms}")

Calibration multispectrum completed: image_reduced.ms.fits
